# professor에게 듣는 연구 한마디

VSCode에서 `professor` 스킬로 답변을 만들고, 그 답변을 OmniVoice로 읽어봅니다.

**질문 → 스킬로 답변 만들기 → 답변 붙여넣기 → 음성 합성 → 듣고 저장하기**

위에서부터 설명을 읽고 코드 셀을 하나씩 실행하세요. `Shift + Enter`로 실행할 수 있습니다.
직접 바꿀 곳은 **음성 파일 경로**, **질문**, **답변 텍스트**입니다.
언어·스타일·속도·길이·추론 단계는 별도로 지정하지 않습니다.

준비물은 CUDA를 사용할 수 있는 NVIDIA GPU, Python 3.10 이상 환경, 그리고 저장소에 포함된 `voice_sample.mp3`입니다.
VSCode에서는 이 저장소 폴더를 열어 `professor` 스킬을 사용할 수 있어야 합니다.
답변은 에이전트에서 생성하므로 이 노트북에 LLM API 키를 넣을 필요가 없습니다.

Colab을 사용할 경우 [런타임 이용 범위](https://research.google.com/colaboratory/faq.html)를 확인하세요.
음성 합성은 해당 이용이 허용되는 환경에서 진행합니다. 같은 노트북을 학교 GPU나 로컬 Jupyter에서도 사용할 수 있습니다.

## 1. 실행 환경 확인하기

먼저 지금 코드를 실행하는 컴퓨터와 GPU를 확인합니다. 화면이 VSCode여도 커널은 다른 서버에 있을 수 있습니다.

- Colab에서는 런타임의 하드웨어 가속기를 GPU로 선택합니다.
- 학교 서버나 로컬 PC에서는 CUDA용 PyTorch와 torchaudio가 설치된 커널을 선택합니다.
- GPU 이름과 사용 가능한 메모리가 나오면 다음 셀로 넘어갑니다. GPU가 없으면 이 셀에서 멈춰 환경부터 바꿉니다.

In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 이상 커널을 선택해 주세요.")

try:
    import torch
    import torchaudio
except (ImportError, OSError) as exc:
    raise RuntimeError(
        "CUDA용 PyTorch와 torchaudio가 설치된 커널을 선택해 주세요. "
        "로컬 설치는 https://pytorch.org/get-started/locally/ 를 참고하세요."
    ) from exc

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 연결되지 않았습니다. GPU 커널을 선택한 뒤 다시 실행해 주세요.")

free_bytes, total_bytes = torch.cuda.mem_get_info()
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU 메모리: 여유 {free_bytes / 2**30:.1f} / 전체 {total_bytes / 2**30:.1f} GiB")
print("작업 폴더:", Path.cwd())

WORK_DIR = Path.cwd() / "_professor_lab"
WORK_DIR.mkdir(exist_ok=True)

## 2. 필요한 패키지 설치하기

처음 한 번 실행합니다. 수업 중 코드가 달라지는 것을 줄이기 위해 OmniVoice 소스 버전을 고정합니다.
기존 PyTorch와 torchaudio는 유지합니다. 다운로드 중에는 셀이 끝날 때까지 기다려 주세요.

설치가 끝난 뒤 모듈을 가져오는 오류가 생기면 커널을 재시작하고 1번부터 다시 실행합니다.

In [ ]:
import subprocess

OMNIVOICE_REVISION = "08be0b4ccbac3e13e374e86fbfead4b4cac343e2"
constraints_file = WORK_DIR / "torch-constraints.txt"
constraints_file.write_text(
    f"torch=={version('torch')}\ntorchaudio=={version('torchaudio')}\n",
    encoding="utf-8",
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--constraint", str(constraints_file),
    f"git+https://github.com/k2-fsa/OmniVoice.git@{OMNIVOICE_REVISION}",
    "librosa==0.11.0", "soundfile==0.13.1", "imageio-ffmpeg==0.6.0",
])
print("설치가 끝났습니다.")

## 3. 참조 음성 준비하기

저장소에 포함된 `voice_sample.mp3`를 사용합니다. 아래 `REFERENCE_AUDIO`를 파일 경로로 바꿀 수 있습니다.
현재 폴더와 가까운 상위 폴더의 `private/`, `prof/`에서도 같은 파일을 찾습니다.

- **로컬 VSCode/Jupyter:** 파일을 찾지 못하면 경로를 직접 입력합니다.
- **브라우저 Colab:** 기본 샘플이 없으면 저장소에서 자동으로 가져옵니다. 별도 업로드는 필요하지 않습니다.
- **VSCode의 원격 커널:** 로컬 PC 경로가 그대로 보이지 않습니다. 커널이 실행되는 서버에 파일을 올리고 서버 경로를 입력합니다. 업로드 창을 지원하지 않으면 브라우저 Colab에서 업로드하세요.

기본 샘플 대신 다른 파일을 쓰려면 `REFERENCE_AUDIO`에 해당 경로를 입력하세요.

In [ ]:
REFERENCE_AUDIO = "voice_sample.mp3"  # 필요하면 이 값만 파일 경로로 바꿉니다.

def locate_reference(value):
    requested = Path(value).expanduser()
    if requested.is_absolute():
        if requested.is_file():
            return requested.resolve()
        raise FileNotFoundError("입력한 음성 파일 경로를 찾을 수 없습니다.")

    candidates = []
    for base in [Path.cwd(), *list(Path.cwd().parents)[:2]]:
        candidates.extend([base / requested, base / "private" / requested, base / "prof" / requested])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    if requested == Path("voice_sample.mp3"):
        from urllib.request import urlopen
        import hashlib

        sample_url = "https://raw.githubusercontent.com/jumincho/AI-JBNU-2026/main/voice_sample.mp3"
        destination = WORK_DIR / "voice_sample.mp3"
        print("저장소에서 참조 음성을 가져옵니다.")
        try:
            with urlopen(sample_url, timeout=30) as response:
                data = response.read()
        except OSError as exc:
            raise RuntimeError(
                "참조 음성을 내려받지 못했습니다. 인터넷 연결을 확인하거나 "
                "저장소의 voice_sample.mp3를 직접 받아 REFERENCE_AUDIO 경로를 지정해 주세요."
            ) from exc
        expected_sha256 = "f336df1c295e8d3e29860217e7e1381d8306e803ee6bb4a15a81af3191c3c05f"
        if hashlib.sha256(data).hexdigest() != expected_sha256:
            raise RuntimeError("받은 샘플이 예상한 파일과 다릅니다. 저장소의 최신 노트북을 사용해 주세요.")
        destination.write_bytes(data)
        return destination.resolve()

    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "voice_sample.mp3를 찾지 못했습니다. REFERENCE_AUDIO를 실제 파일 경로로 바꿔 주세요."
        ) from exc

    print("voice_sample.mp3 파일 하나를 선택해 주세요.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("음성 파일 하나만 선택한 뒤 이 셀을 다시 실행해 주세요.")
    filename, blob = next(iter(uploaded.items()))
    if Path(filename).suffix.lower() not in {".mp3", ".wav", ".flac", ".m4a"}:
        raise ValueError("MP3, WAV, FLAC, M4A 중 하나를 사용해 주세요.")
    destination = WORK_DIR / ("uploaded_reference" + Path(filename).suffix.lower())
    destination.write_bytes(blob)
    return destination.resolve()

reference_path = locate_reference(REFERENCE_AUDIO)
print("음성 파일을 찾았습니다:", reference_path.name)

## 4. 음성을 읽고 들어보기

MP3를 읽어 모델이 다루기 쉬운 WAV로 준비합니다. 별도로 자르거나 음량을 바꾸지 않습니다.
이후 OmniVoice가 수행하는 기본 전처리는 그대로 둡니다.

아래 재생 버튼으로 다른 파일을 고르지 않았는지 확인하세요. 공식 권장 참조 길이는 3~10초이며,
준비된 샘플은 약 7초입니다. 파일 읽기가 끝나면 음성 길이와 재생 버튼이 나타납니다.

In [ ]:
import hashlib
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def decode_reference(source, destination):
    try:
        data, sample_rate = sf.read(str(source), dtype="float32", always_2d=True)
    except (RuntimeError, sf.LibsndfileError):
        import imageio_ffmpeg
        intermediate = destination.parent / "decoded_reference.wav"
        subprocess.run([
            imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-loglevel", "error",
            "-i", str(source), "-vn", "-c:a", "pcm_f32le", str(intermediate),
        ], check=True)
        data, sample_rate = sf.read(str(intermediate), dtype="float32", always_2d=True)
    if len(data) == 0 or not np.isfinite(data).all() or not np.any(data):
        raise ValueError("음성이 비어 있거나 읽을 수 없습니다. 파일을 확인해 주세요.")
    sf.write(str(destination), data, sample_rate, subtype="FLOAT")
    return data, sample_rate

reference_wav = WORK_DIR / "reference.wav"
reference_data, reference_rate = decode_reference(reference_path, reference_wav)
reference_seconds = len(reference_data) / reference_rate
sample_key = hashlib.sha256(reference_path.read_bytes()).hexdigest()
if globals().get("_sample_key") != sample_key:
    globals().pop("voice_prompt", None)
    globals().pop("last_result", None)
_sample_key = sample_key

print(f"참조 음성: {reference_seconds:.2f}초 / {reference_rate:,} Hz / {reference_data.shape[1]}채널")
if not 3 <= reference_seconds <= 10:
    print("참고: 공식 권장 참조 길이는 3~10초입니다.")
display(Audio(filename=str(reference_wav)))

## 5. OmniVoice 불러오기

모델은 Hugging Face에서 받아 GPU에 올립니다. 첫 실행은 모델 다운로드 때문에 시간이 더 걸립니다.
같은 커널에서 답변을 바꿔 읽을 때는 모델을 다시 불러오지 않습니다.

`float16`은 GPU에서 모델을 실행하는 방식입니다. 언어·말하기 속도 같은 생성 옵션을 바꾸는 설정은 아닙니다.

In [ ]:
from omnivoice import OmniVoice

MODEL_ID = "k2-fsa/OmniVoice"
if globals().get("_loaded_model_id") != MODEL_ID or "model" not in globals():
    model = OmniVoice.from_pretrained(
        MODEL_ID,
        device_map="cuda:0",
        dtype=torch.float16,
        load_asr=True,
    )
    _loaded_model_id = MODEL_ID
    globals().pop("voice_prompt", None)
    globals().pop("last_result", None)
    print("모델을 불러왔습니다.")
else:
    print("이미 불러온 모델을 사용합니다.")

## 6. 같은 목소리를 다시 쓸 준비하기

참조 음성을 한 번 처리해 두면 새 답변을 읽을 때 재사용할 수 있습니다.
참조 전사문도 직접 입력하지 않고 자동으로 읽게 둡니다.
언어·스타일·전처리 옵션은 지정하지 않습니다.

다른 참조 음성으로 바꿨다면 3번부터 이 셀까지 다시 실행하세요.

In [ ]:
if "voice_prompt" not in globals():
    with torch.inference_mode():
        voice_prompt = model.create_voice_clone_prompt(ref_audio=str(reference_wav))
    print("목소리 준비가 끝났습니다.")
else:
    print("준비해 둔 목소리를 다시 사용합니다.")

## 7. professor에게 질문하기

아래 `QUESTION`을 바꾸고 셀을 실행하세요. Codex용과 Claude Code용 요청이 출력됩니다.
사용 중인 에이전트에 해당 요청을 복사해서 보내세요.

이 단계는 **VSCode의 에이전트 대화창**에서 진행합니다. 브라우저 Colab에서 노트북을 열었더라도
에이전트는 저장소를 연 VSCode에서 사용합니다. `.ipynb`를 실행하는 것만으로 스킬이 자동 호출되지는 않습니다.

In [ ]:
QUESTION = "교수님, 연구가 너무 하기 싫어요."

if not QUESTION.strip():
    raise ValueError("QUESTION에 질문을 적어 주세요.")
request_body = (
    "음성용으로 한국어 구어체 2~4문장만 답해주세요. "
    "제목이나 목록 없이 읽을 말만 써주세요.\n질문: " + QUESTION.strip()
)
print("[Codex에 보낼 요청]\n$professor " + request_body)
print("\n[Claude Code에 보낼 요청]\n/professor " + request_body)

## 8. 생성된 답변 붙여넣기

에이전트가 만든 **답변만** 아래 따옴표 안에 붙여넣으세요. 질문·스킬 호출 명령·설명은 빼고 읽을 문장만 넣습니다.
여러 줄로 붙여넣어도 됩니다. 처음에는 짧은 2~4문장으로 시작하세요.

답변을 비워두면 이 셀에서 입력을 안내합니다. 새 답변을 받아보려면 7~10번을 다시 실행하면 됩니다.

In [ ]:
REPLY_TEXT = """
"""  # 이 따옴표 안에 professor가 만든 답변을 붙여넣습니다.

def prepare_reply(text):
    import textwrap
    value = textwrap.dedent(text).strip()
    if not value:
        raise ValueError("professor가 만든 답변을 REPLY_TEXT 따옴표 안에 붙여넣어 주세요.")
    return value

reply_text = prepare_reply(REPLY_TEXT)
globals().pop("last_result", None)
print(f"읽을 답변 ({len(reply_text)}자):\n{reply_text}")
if len(reply_text) > 400:
    print("첫 실습은 조금 짧게 시작하면 기다리는 시간을 줄일 수 있습니다.")

## 9. 음성 합성하기

준비된 목소리로 답변을 읽습니다. 언어·스타일·속도·길이·추론 단계·텍스트 정규화 옵션은 넣지 않습니다.
같은 텍스트라도 실행할 때마다 결과가 조금 달라질 수 있습니다.

완료되면 처리 시간과 생성된 음성 길이가 나옵니다. GPU 메모리가 부족하면 다른 GPU 작업을 닫거나 답변을 짧게 해보세요.

In [ ]:
import time
from datetime import datetime

if "reply_text" not in globals() or not reply_text:
    raise RuntimeError("8번 셀에서 답변을 먼저 입력해 주세요.")
if "voice_prompt" not in globals():
    raise RuntimeError("6번 셀에서 목소리를 먼저 준비해 주세요.")

globals().pop("last_result", None)
torch.cuda.empty_cache()
torch.cuda.synchronize()
started = time.perf_counter()
try:
    with torch.inference_mode():
        generated = model.generate(text=reply_text, voice_clone_prompt=voice_prompt)
    torch.cuda.synchronize()
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError("GPU 메모리가 부족합니다. 다른 작업을 닫거나 답변을 줄인 뒤 다시 실행해 주세요.") from exc

waveform = np.asarray(generated[0], dtype=np.float32).reshape(-1)
if not len(waveform) or not np.isfinite(waveform).all():
    raise RuntimeError("유효한 음성이 생성되지 않았습니다. 같은 셀을 다시 실행해 주세요.")

output_rate = int(model.sampling_rate)
run_name = datetime.now().strftime("professor_%Y%m%d_%H%M%S_%f")
output_wav = WORK_DIR / f"{run_name}.wav"
output_text = WORK_DIR / f"{run_name}.txt"
sf.write(str(output_wav), waveform, output_rate, subtype="PCM_16")
output_text.write_text(reply_text, encoding="utf-8")
last_result = {"wav": output_wav, "text_file": output_text, "reply": reply_text, "sample_key": _sample_key}
print(f"합성 완료: 처리 {time.perf_counter() - started:.1f}초 / 음성 {len(waveform) / output_rate:.1f}초")
torch.cuda.empty_cache()

## 10. 들어보고 저장하기

재생 버튼으로 답변을 들어보세요. 파일 링크를 누르면 WAV를 받을 수 있습니다.
질문에 맞는 내용인지, 읽는 흐름이 자연스러운지 함께 살펴봅니다.

다른 답변을 읽으려면 7~10번을 다시 실행합니다. 목소리와 모델은 그대로 재사용합니다.

In [ ]:
from IPython.display import FileLink

def current_result():
    result = globals().get("last_result")
    if not result or result["reply"] != globals().get("reply_text") or result["sample_key"] != _sample_key:
        raise RuntimeError("현재 답변으로 9번 음성 합성을 먼저 실행해 주세요.")
    return result

result = current_result()
display(Audio(filename=str(result["wav"])))
display(FileLink(str(result["wav"])))

## 11. 답변과 음성을 함께 받기

이 셀은 방금 만든 WAV와 답변 TXT를 ZIP 하나로 묶습니다. 참조 음성은 ZIP에 넣지 않습니다.
브라우저 Colab에서는 다운로드가 시작되고, 다른 환경에서는 파일 링크를 사용하면 됩니다.

노트북을 공유할 때는 실행 출력을 지우고 저장하세요. 오디오 재생 출력에도 음성이 포함될 수 있습니다.

In [ ]:
import zipfile

result = current_result()
archive = result["wav"].with_suffix(".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in [result["wav"], result["text_file"]]:
        bundle.write(file, arcname=file.name)

try:
    from google.colab import files
except ImportError:
    display(FileLink(str(archive)))
else:
    files.download(str(archive))
print("답변과 음성을 묶었습니다. 참조 음성은 포함하지 않았습니다.")

## 조금 더 해보기

1. 같은 고민을 일반 답변과 `professor` 스킬 답변으로 각각 만들어 비교해 보세요.
2. 같은 내용을 짧은 문장과 긴 문장으로 바꿔 읽어보세요. 어느 쪽이 듣기 편한가요?
3. 같은 텍스트로 9번만 다시 실행해 보세요. 목소리와 억양이 얼마나 달라지나요?

| 막힌 곳 | 먼저 확인할 것 |
| --- | --- |
| GPU가 없다고 나와요 | 현재 선택한 커널과 CUDA 연결을 확인합니다. |
| 설치 후 import 오류가 나요 | 커널을 재시작하고 1번부터 실행합니다. PyTorch와 torchaudio 버전이 맞는지 확인합니다. |
| 파일을 찾지 못해요 | 로컬 경로와 원격 커널의 경로가 다를 수 있습니다. 실행 서버에서 보이는 경로를 넣습니다. |
| 첫 실행이 오래 걸려요 | 음성 모델과 자동 전사 모델을 처음 내려받는 중일 수 있습니다. |
| 메모리가 부족해요 | 다른 GPU 작업을 닫고 답변을 줄입니다. 계속되면 커널을 재시작하고 모델부터 다시 준비합니다. |
| 목소리를 바꿨는데 이전 결과가 들려요 | 3~6번을 다시 실행한 뒤 새로 합성합니다. |

사용법을 더 보고 싶다면 [OmniVoice](https://github.com/k2-fsa/OmniVoice)와
[모델 카드](https://huggingface.co/k2-fsa/OmniVoice)를 참고하세요.